# Get to Know a Dataset: NUVIEW State Open Data

This notebook serves as a guided tour of the [NUVIEW State Open Data](https://registry.opendata.aws/nuview-state-opendata) dataset. More usage examples, tutorials, and documentation for this dataset and others can be found at the [Registry of Open Data on AWS](https://registry.opendata.aws/).

At the top level of our S3 bucket, we have multiple key prefixes, each key prefix defines US state or agency that the data was collected for as well as the type of remote sensing product.
For example `ak_alaska_bare_earth` key prefix contains LIDAR derived product of bare earth surface collected across the state of Alaska.

Underneath the top level key prefix the data is organized into acqusition projects. Project names follow state or agency naming conventions. Each project represents single acqusition contract between data proider and state or agency representatives.

For example under `ak_alaska_bare_earth` key prefix there are several projects:

- AGO_EVOS_Copper_River_2023/
- fairbanks_unalakleet_nv5_09_sep_2024/

Under each project the files are organized according to the USGS data collection guidelines (ex. [Lidar Base Specification](https://www.usgs.gov/ngp-standards-and-specifications/lidar-base-specification-online) for collections under the 3D Elevation Program).
 
Full documentation for this dataset can be found at: https://github.com/s22s/nuview-state-opendata




## How to access NUVIEW State Open Data 

First we will import the Python libraries required throughout this notebook.


In [ ]:
# This notebook requires the following additional libraries
# (please install using the preferred method for your environment, e.g. pip, conda):
#
# boto3 >= 1.38.23
# polars >= 1.30.0
# matplotlib >= 3.10.3 
# numpy >= 1.21.0
# pdal
# rasterio
# pyvista[jupyter]

# Import the libraries required for this notebook

# Built-ins
import json
import tempfile
from pprint import pprint

# Installed libraries
import pdal
import numpy as np
import pyvista as pv
import mercury as mr

import matplotlib.pyplot as plt
from matplotlib import cm

import rasterio
from rasterio.io import MemoryFile
from rasterio.plot import show

import boto3
from botocore import UNSIGNED
from botocore.config import Config

Next, we will define the location of our dataset, create our boto3 S3 client, and list the top level prefixes in our S3 bucket. Here we see there is only one top-level prefix in our bucket.

In [ ]:
# Location of the S3 bucket for this dataset
bucket = "nuview-state-opendata"

# List the top level of the bucket using boto3. Because this is a public bucket, we don't need to sign requests.
# Here we set the signature version to unsigned, which is required for public buckets.
s3 = boto3.client('s3', config=Config(signature_version=UNSIGNED))

# Print the items in the top-level prefixes
for item in s3.list_objects_v2(Bucket=bucket, Delimiter='/')['CommonPrefixes']:
    print(item['Prefix'])

Looking into the top-level S3 prefix of our dataset, we see that the data have been separated into datasets based on location (state) and product type.

In [ ]:
# List the key prefixes within the top level 'ak_alaska_bare_earth/' prefix
for item in s3.list_objects_v2(Bucket=bucket, Prefix='ak_alaska_bare_earth/', Delimiter='/', MaxKeys=10)['CommonPrefixes']:
    print(item['Prefix'])

Listing key prefixes under `ak_alaska_bare_earth` reveales several project names.
Let's look at `AK_Copper_River_FINAL`. 

In [ ]:
# List the keys within the 'ak_alaska_bare_earth/AK_Copper_River_FINAL/' prefix.
for item in s3.list_objects_v2(Bucket=bucket, Prefix='ak_alaska_bare_earth/AK_Copper_River_FINAL/', MaxKeys=10)['Contents']:
    print(item['Key'])

The `bare_earth/be_rasters` key prefix follows the USGS [Lidar Base Specification](https://www.usgs.gov/ngp-standards-and-specifications/lidar-base-specification-online) for collections under the 3D Elevation Program. 
Next the files are organized based on the UTM zone they were collected in. Finally the file name consist of the UTM6 grid cell coordinates and the acquisition date.
All raster files are Cloud Optimized GeoTiffs.

Let's take a look at one of the files.

In [ ]:
# First we'll load the data into memory and create an elevation image.

file_key = "ak_alaska_bare_earth/AGO_EVOS_Copper_River_2023/bare_earth/be_rasters/utm_zone_06/UTM6_0315_0807_4_2024.cog.tiff"

with s3.get_object(Bucket=bucket, Key=file_key)['Body'] as data:
    with MemoryFile(data.read()) as memfile:
     with memfile.open() as src:
        dem = src.read(1)

        # Show as an image with 'terrain' color map
        plt.imshow(dem, cmap="terrain")
        plt.colorbar(label="Elevation")
        plt.show()


Now let's add hill shade.

In [ ]:
# Simple hill shading algorithm
def hillshade(array, azimuth=315, altitude=45):
    x, y = np.gradient(array)
    slope = np.pi/2 - np.arctan(np.sqrt(x*x + y*y))
    aspect = np.arctan2(-x, y)
    az = np.radians(azimuth)
    alt = np.radians(altitude)
    shaded = np.sin(alt)*np.sin(slope) + np.cos(alt)*np.cos(slope)*np.cos(az-aspect)
    return (shaded + 1) / 2

with s3.get_object(Bucket=bucket, Key=file_key)['Body'] as data:
    with MemoryFile(data.read()) as memfile:
     with memfile.open() as src:
        dem = src.read(1)
        dem_norm = (dem - np.nanmin(dem)) / (np.nanmax(dem) - np.nanmin(dem))
        hs = hillshade(dem)
        
        # Multiply colormap RGB by hillshade for terrain effect
        rgb = cm.terrain(dem_norm)[:, :, :3]
        rgb_hs = rgb * hs[..., np.newaxis]

        # Show as an image with hill shade and 'terrain' color map
        plt.imshow(rgb_hs)
        plt.colorbar(label="Elevation")
        plt.show()

## Accessing NUVIEW State Open Data LIDAR point cloud data

Let's look at the original lidar point cloud file that this DEM was derived from.
As before the key prefix structure follows the USGS [Lidar Base Specification](https://www.usgs.gov/ngp-standards-and-specifications/lidar-base-specification-online) for collections under the 3D Elevation Program so we need to look for point cloud files under `ak_alaska_pointcloud/AGO_EVOS_Copper_River_2023/point_cloud`.
Let's list the contents of that prefix:

In [ ]:
# List items under 'ak_alaska_pointcloud/AGO_EVOS_Copper_River_2023/point_cloud' prefix:
for item in s3.list_objects_v2(Bucket=bucket, Prefix='ak_alaska_pointcloud/AGO_EVOS_Copper_River_2023/point_cloud', MaxKeys=10)['Contents']:
    print(item['Key'])

All point cloud files in the dataset are LAZ version 1.4 with [COPC](https://copc.io) layout.

Let's take a look at one of the LAZ files:

In [ ]:
# Let's get the file first and keep as temporary file for further processing.
file_key='ak_alaska_pointcloud/AGO_EVOS_Copper_River_2023/point_cloud/tilecls/Orthometric/utm_zone_06/UTM6_0315_0807_4_2024.copc.laz'

with s3.get_object(Bucket=bucket, Key=file_key)['Body'] as data:
    with tempfile.NamedTemporaryFile() as laz_file:
        laz_file.write(data.read())

        pipeline_json = {
            "pipeline": [
                {
                    "type": "readers.las",
                    "filename": laz_file.name
                }
            ]
        }
        
        pipeline = pdal.Pipeline(json.dumps(pipeline_json))
        pipeline.execute()
        
        # Get metadata (same as pdal info)
        metadata = pipeline.metadata
        mr.JSON(metadata, level=2)

Let's render the point cloud in 3d.

In [ ]:
pts = pipeline.arrays[0]

meta = metadata["metadata"]["readers.las"]
scale = np.array([meta["scale_x"], meta["scale_y"], meta["scale_z"]])
offset = np.array([meta["offset_x"], meta["offset_y"], meta["offset_z"]])

X = pts["X"] * scale[0] + offset[0]
Y = pts["Y"] * scale[1] + offset[1]
Z = pts["Z"] * scale[2] + offset[2]

cloud = np.vstack([X, Y, Z]).T

p = pv.PolyData(cloud)
plotter = pv.Plotter(notebook=True)
plotter.add_points(p, scalars=Z, render_points_as_spheres=False, point_size=3, cmap="terrain")
plotter.show()

Following the 2022 Typhoon Merbok coastal storm, state agencies and community partners used the Alaska Geospatial Office’s high-resolution imagery and elevation data to answer a critical question:
“**Where, and to what elevation, did storm-surge flooding occur in an affected Alaska community?**”

AGO’s orthomosaic imagery and elevation datasets provided the foundation for reconstructing peak inundation. Analysts loaded both datasets into a GIS environment, aligned them to a common projection, and prepared the DEM for hydrologic analysis. Post-storm imagery revealed high-water marks, debris lines, and flooded roads. These features were digitized and sampled against the DEM to estimate water-surface elevations.
Using these elevations, analysts identified all terrain at or below the peak water level and contiguous with the coastline, producing an initial inundation extent based on topography. The modeled area was refined by comparing it to the imagery and adjusting boundaries where dunes, berms, or road embankments influenced flow. The final inundation layer was intersected with infrastructure data to support emergency response, recovery, and long-term planning.
AGO’s imagery showed where the water reached, while AGO’s elevation data quantified how high it rose—together enabling a detailed and defensible reconstruction of storm-surge flooding.

A significant statewide hazard question in Alaska remains largely unanswered:
“**Which glacial lakes pose the greatest potential for future glacial lake outburst floods, and how can imagery and elevation data be used to identify and prioritize these risks?**”

As glaciers retreat, new lakes form and existing lakes expand behind ice or loosely consolidated moraine dams. Some of these lakes can drain suddenly, producing high-energy floods that threaten downstream communities, roads, and ecosystems. Alaska does not yet have a systematic, statewide assessment of these hazards.
AGO’s imagery and elevation products provide a strong foundation for such an effort. A user could identify glacier-proximal lakes from imagery, track their change over time, and estimate lake elevation, basin geometry, and dam height directly from the DEM. Hydrologic modeling on the DEM would enable simulation of potential outburst paths and identification of downstream exposure.

To address this question, users should:
- Use multi-temporal imagery to track lake formation and expansion.
- Combine DEMs with other available elevation sources to refine dam characteristics.
- Apply hydrologic modeling tools to simulate potential breach scenarios.
- Validate results with local knowledge and field observations.
- Document uncertainties in dam material, DEM accuracy, and imagery timing.
- Automate broad-scale processing for statewide screening while reserving manual review for high-priority sites.

Answering this question would provide Alaska’s first consistent, statewide view of glacial-lake hazards. AGO’s imagery and elevation datasets supply the essential inputs to make that possible.